# Building the Votes Table

This notebook fetches Senate roll call votes from the congress.gov API and builds a
long-format votes table with one row per (senator, bill, vote). Vote values from the API are: `Yes`, `No`, `Not Voting`, `Present`


In [2]:
# imports
import os
import time
import requests
import pandas as pd

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time

# congress range to fetch — senate.gov has data back to the 101st (1989)
# each congress has 2 sessions
CONGRESSES = range(101, 120)  # 101st through 119th
DELAY = 0.3

def get_vote_list(congress, session):
    """Fetch the list of all roll call votes for one congress+session."""
    url = f'https://www.senate.gov/legislative/LIS/roll_call_lists/vote_menu_{congress}_{session}.xml'
    resp = requests.get(url, timeout=15)
    if resp.status_code == 404:
        return []
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    votes = []
    for item in root.findall('.//vote'):
        vote_number = item.findtext('vote_number')
        issue       = item.findtext('issue')       # bill id like "S. 1234"
        question    = item.findtext('question')    # "On Passage", "On Cloture", etc.
        if vote_number:
            votes.append({
                'congress':     congress,
                'session':      session,
                'vote_number':  vote_number.zfill(5),
                'issue':        issue,
                'question':     question
            })
    return votes


def get_member_votes(congress, session, vote_number):
    """Fetch per-senator vote breakdown for one roll call."""
    num = str(vote_number).zfill(5)
    url = (
        f'https://www.senate.gov/legislative/LIS/roll_call_votes/'
        f'vote{congress}{session}/vote_{congress}_{session}_{num}.xml'
    )
    resp = requests.get(url, timeout=15)
    if resp.status_code == 404:
        return []
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    members = []
    for m in root.findall('.//member'):
        bioguide = m.findtext('bioguide_id') or m.findtext('lis_member_id')
        vote_cast = m.findtext('vote_cast')
        if bioguide:
            members.append({'senator_id': bioguide, 'vote': vote_cast})
    return members


rows = []

for congress in CONGRESSES:
    for session in [1, 2]:
        print(f'Congress {congress}, session {session}...')
        vote_list = get_vote_list(congress, session)

        # only keep votes on actual bills (question is "On Passage" etc.)
        bill_votes = [
            v for v in vote_list
            if v['question'] and 'passage' in v['question'].lower()
        ]
        print(f'  {len(bill_votes)} passage votes')

        for v in bill_votes:
            member_votes = get_member_votes(congress, session, v['vote_number'])
            for mv in member_votes:
                rows.append({
                    'senator_id': mv['senator_id'],
                    'bill_id':    v['issue'],
                    'vote':       mv['vote'],
                    'congress':   congress
                })
            time.sleep(DELAY)

votes_df = pd.DataFrame(rows, columns=['senator_id', 'bill_id', 'vote', 'congress'])
print(f'\nDone. {len(votes_df)} rows, {votes_df["senator_id"].nunique()} senators, {votes_df["bill_id"].nunique()} bills')

Congress 101, session 1...
  30 passage votes
Congress 101, session 2...
  34 passage votes
Congress 102, session 1...
  30 passage votes
Congress 102, session 2...
  33 passage votes
Congress 103, session 1...
  32 passage votes
Congress 103, session 2...
  30 passage votes
Congress 104, session 1...
  36 passage votes
Congress 104, session 2...
  26 passage votes
Congress 105, session 1...
  28 passage votes
Congress 105, session 2...
  29 passage votes
Congress 106, session 1...
  29 passage votes
Congress 106, session 2...
  26 passage votes
Congress 107, session 1...
  30 passage votes
Congress 107, session 2...
  18 passage votes
Congress 108, session 1...
  34 passage votes
Congress 108, session 2...
  19 passage votes
Congress 109, session 1...
  25 passage votes
Congress 109, session 2...
  23 passage votes
Congress 110, session 1...
  32 passage votes
Congress 110, session 2...
  18 passage votes
Congress 111, session 1...
  30 passage votes
Congress 111, session 2...
  10 pa

OSError: Cannot save file into a non-existent directory: '../data/votes'

In [6]:
# save to csv
votes_df.to_csv('../../data/votes/votes.csv', index=False)